# SAD Classification (binary): Tangent-Space Logistic Regression

## TLDR

This notebook uses three outer subject-disjoint folds for evaluation. Inside each outer training set, a second three-fold search chooses between two tangent-space metrics and three logistic-regression regularization strengths. The outer test subjects are never used to choose parameters.

## Context & Methods

We ask a simple question: can resting-state EEG distinguish subjects with a recorded social anxiety disorder (SAD) diagnosis from those without one? The second group is not necessarily healthy; its subjects may have other diagnoses. You can run the same binary classifier for another disorder by changing `TARGET_NAME` and `TARGET_COLUMN` in the parameters cell — for example, Depression (`SCID5_CV_Depression`) or any other diagnosis column in `labels_reduced.csv`.

We use this pipeline:

4–15 Hz filter → 2 s windows → OAS covariance → per-state mean covariance → tangent space → concatenation → standardization → logistic regression

For each subject, we average the window covariances separately for the eyes-open and eyes-closed recordings. We then map the two matrices to tangent space, combine their features, and fit logistic regression. Balanced accuracy is the selection and evaluation metric. The preprocessing choices are fixed before looking at the labels; the tangent metric and logistic-regression strength are selected in nested, subject-disjoint cross-validation.

In [1]:
import os
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from pcp_project.classify import (
    StateTangentFeatures,
    build_classification_pipeline,
    cross_validate_subjects,
    make_subject_folds,
    nested_search_subjects,
)
from pcp_project.data import (
    binary_target,
    list_subject_ids,
    load_labels,
    load_subject,
)

In [2]:
RANDOM_STATE = 7
N_SPLITS = 3
INNER_SPLITS = 3
TARGET_NAME = "SAD"
TARGET_COLUMN = "SCID5_CV_SAD"

WINDOW_SECONDS = 2.0
FREQUENCY_BANDS = ((4.0, 15.0),)
TANGENT_METRICS = ("riemann", "logeuclid")
LOGISTIC_C_VALUES = (0.01, 0.1, 1.0)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pcp_project").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(
    os.environ.get("PCP_DATA_DIR", PROJECT_ROOT / "data" / "data-20260528")
)

## Data

We keep every EEG recording that has a row in **labels_reduced.csv**. Each subject appears exactly once.

In [3]:
labels = load_labels(DATA_DIR / "labels_reduced.csv")
available_eeg_ids = list_subject_ids(DATA_DIR)
subject_ids = [subject_id for subject_id in available_eeg_ids if subject_id in labels.index]

y = binary_target(labels, subject_ids, TARGET_COLUMN)
groups = np.asarray(subject_ids)
target_labels = np.where(
    y == 1, f"{TARGET_NAME} recorded", f"{TARGET_NAME} not recorded"
)

assert len(subject_ids) == len(y) == len(np.unique(groups))
assert set(np.unique(y)) == {0, 1}

print(f"Matched subjects: {len(subject_ids)} of {len(available_eeg_ids)} EEG files")
pd.Series(target_labels, name="target group").value_counts().to_frame("subjects")

Matched subjects: 44 of 44 EEG files


,subjects
target group,
SAD not recorded,23
SAD recorded,21


### Prepare state covariances

Loading all raw recordings at once would use more than 10 GB before windowing. Instead, the project pipeline builder runs the fixed preprocessing one subject at a time and averages each state's window covariances. State runs stay separate, so no window crosses an eyes-open/eyes-closed boundary. Each subject reduces to one eyes-open and one eyes-closed covariance matrix.

We keep the 4–15 Hz band and non-overlapping 2 s windows fixed. At 256 Hz, a window has 512 samples for the 61-channel covariance; this is a practical compromise between covariance support and the number of windows. It is also close to a published resting-state EEG setup using non-overlapping 2 s epochs and sub-bands covering 4–16 Hz ([study](https://www.mdpi.com/2076-3417/13/11/6606)). That precedent does not prove this band is optimal for this cohort, so the notebook states it as a modelling choice rather than a tuned result.

OAS shrinkage is used because each covariance is estimated from a short window in a moderately high-dimensional setting. The estimator was derived for improved covariance estimation when the sample covariance is noisy ([Chen et al., 2010](https://arxiv.org/abs/0907.4698)). These operations are label-free and subject-local, so computing them before the nested split does not share information across subjects.

In [4]:
covariance_extractor = build_classification_pipeline(
    classifier="passthrough",
    window_seconds=WINDOW_SECONDS,
    frequency_bands=FREQUENCY_BANDS,
    covariance="oas",
    aggregation="stack",
    state_order=(0, 1),
    feature_transform="identity",
)

In [5]:
preprocessing_start = perf_counter()

first_subject = load_subject(subject_ids[0], DATA_DIR)
covariance_extractor.fit([first_subject])
covariance_rows = [covariance_extractor.transform([first_subject])[0]]
del first_subject

for position, subject_id in enumerate(subject_ids[1:], start=2):
    subject = load_subject(subject_id, DATA_DIR)
    covariance_rows.append(covariance_extractor.transform([subject])[0])
    del subject
    if position % 10 == 0 or position == len(subject_ids):
        print(f"Prepared {position}/{len(subject_ids)} subjects")

subject_state_covariances = np.stack(covariance_rows)

print(f"Subject/state covariance tensor: {subject_state_covariances.shape}")
print(f"Preprocessing time: {perf_counter() - preprocessing_start:.1f} seconds")

Prepared 10/44 subjects
Prepared 20/44 subjects
Prepared 30/44 subjects
Prepared 40/44 subjects
Prepared 44/44 subjects
Subject/state covariance tensor: (44, 2, 61, 61)
Preprocessing time: 71.1 seconds


## Cross-validation

Each subject is a pair of per-state covariance matrices. A scikit-learn pipeline maps each state to tangent space, concatenates the two feature vectors, standardizes, and fits logistic regression. In each outer fold, `nested_search_subjects` creates new inner subject folds using only the outer training rows. The fitted tangent maps, scaler, and classifier therefore never see an outer test subject. This separation matters because selecting parameters and reporting performance on the same folds can bias the estimate ([Cawley & Talbot, 2010](https://www.jmlr.org/papers/v11/cawley10a.html)).

### The classifier pipeline

Each subject arrives as two covariance matrices (eyes open and eyes closed). The pipeline turns those into one prediction in three steps:

1. **Tangent space** — A covariance matrix is not a normal flat feature vector. Tangent space is a standard way to map it to numbers that logistic regression can use; Riemannian covariance methods are well established for EEG classification ([Barachant et al., 2012](https://pubmed.ncbi.nlm.nih.gov/22010143/)). We search `riemann` and `logeuclid`, two standard positive-definite-matrix metrics, separately inside each outer training set.
2. **Standardization** — Rescale features so no single channel pair dominates just because it has larger values.
3. **Logistic regression** — A simple binary classifier. `class_weight="balanced"` gives equal importance to both diagnosis groups. Scikit-learn defines `C` as the inverse regularization strength ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)); the small grid `(0.01, 0.1, 1.0)` spans two orders of magnitude without turning a 44-subject exercise into a large fishing expedition.

In [6]:
folds = make_subject_folds(
    y,
    groups,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
)

fold_rows = []
for fold_number, (train_index, test_index) in enumerate(folds, start=1):
    assert set(groups[train_index]).isdisjoint(groups[test_index])
    fold_rows.append(
        {
            "fold": fold_number,
            "subjects in training fold": len(train_index),
            "subjects in test fold": len(test_index),
            f"{TARGET_NAME} recorded in training fold": int(y[train_index].sum()),
            f"{TARGET_NAME} recorded in test fold": int(y[test_index].sum()),
        }
    )

pd.DataFrame(fold_rows).set_index("fold")

,subjects in training fold,subjects in test fold,SAD recorded in training fold,SAD recorded in test fold
fold,,,,
1,29,15,14,7
2,29,15,14,7
3,30,14,14,7


In [7]:
model = make_pipeline(
    StateTangentFeatures(),
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        solver="liblinear",
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
)
parameter_grid = {
    "statetangentfeatures__metric": TANGENT_METRICS,
    "logisticregression__C": LOGISTIC_C_VALUES,
}

pd.DataFrame(
    {
        "step": list(model.named_steps),
        "estimator": [
            type(estimator).__name__ for estimator in model.named_steps.values()
        ],
    }
).set_index("step")

,estimator
step,
statetangentfeatures,StateTangentFeatures
standardscaler,StandardScaler
logisticregression,LogisticRegression


In [8]:
evaluation_start = perf_counter()

model_scores = nested_search_subjects(
    model,
    parameter_grid,
    subject_state_covariances,
    y,
    groups,
    cv=folds,
    inner_splits=INNER_SPLITS,
    scoring="balanced_accuracy",
    n_jobs=1,
)
dummy_scores = cross_validate_subjects(
    DummyClassifier(strategy="most_frequent"),
    np.zeros((len(y), 1)),
    y,
    groups,
    cv=folds,
    scoring="balanced_accuracy",
    n_jobs=1,
)

print(f"Cross-validation time: {perf_counter() - evaluation_start:.1f} seconds")

Cross-validation time: 5.8 seconds


## Results

Outer-fold balanced accuracy is the primary result because it gives the two diagnosis classes equal importance. `best inner balanced accuracy` is shown only to make the selection process visible; it is not a held-out performance estimate.

In [9]:
fold_results = pd.DataFrame(
    {
        "outer balanced accuracy": model_scores["test_score"],
        "best inner balanced accuracy": model_scores["best_inner_score"],
        "dummy balanced accuracy": dummy_scores["test_score"],
    },
    index=[f"Fold {fold}" for fold in range(1, N_SPLITS + 1)],
)
fold_results["selected tangent metric"] = [
    params["statetangentfeatures__metric"]
    for params in model_scores["best_params"]
]
fold_results["selected C"] = [
    params["logisticregression__C"] for params in model_scores["best_params"]
]
display(fold_results.round(3))

score_columns = [
    "outer balanced accuracy",
    "best inner balanced accuracy",
    "dummy balanced accuracy",
]
fold_results[score_columns].mean().rename("Mean").round(3).to_frame()

,outer balanced accuracy,best inner balanced accuracy,dummy balanced accuracy,selected tangent metric,selected C
Fold 1,0.277,0.525,0.5,logeuclid,0.01
Fold 2,0.402,0.450,0.5,riemann,1.00
Fold 3,0.286,0.572,0.5,riemann,0.01


,Mean
outer balanced accuracy,0.321
best inner balanced accuracy,0.516
dummy balanced accuracy,0.500


## Takeaways

- The metric and `C` shown for a fold were selected using only that fold's training subjects.
- Every outer score is held out: subjects never appear in both training and test within a fold.
- The most-frequent dummy uses the same outer folds, so it remains a direct baseline.